# notes
data i have was collected using `loss()` in `Similarity` featurizer at [src/models/featurizers/similarity.py:192-210]
The loss() method also has a pretraining phase (VAE reconstruction) at lines 174–190


The real featurizer training (to convergence) happens in `eval_featurizer.py:127`, which calls featurizer.train(params["featurizer"]["iterations"], ...) with the full iteration count and pre-loaded human queries.


Feature prediction error — `FeatureLearner` in `eval_featurizer.py`
The measurement is in `feature_learner.py:37`:
```
torch.mean(torch.sum(torch.square(self.feature_learner(self.featurizer.featurize(traj_batch)) - feature_batch), dim=1))
```
It trains a small MLP to predict the human's ground-truth features (human.calc_features(traj)) from the learned featurization, then reports `last_train_loss` and `last_test_loss` — the MSE between predicted and actual features. This is the probe used to measure how much of the human's feature structure the featurizer has captured. It's invoked at `eval_featurizer.py:143-148` after featurizer training is done.

`Similarity` class is what i need, but much simpler

`FeatureLearner` class trains small MLP on top of learned representation to calculate FPE

In [1]:
from utils import *
from sirl import *

anchors, positives, negatives = load_data(200)
model, history = train_sirl(anchors, positives, negatives, use_symmetric_loss=True)

pybullet build time: Oct 21 2025 16:30:45


Epoch    0 | loss=1.9929 | triplet_acc=0.510 | lr=0.00400
Epoch  100 | loss=0.1749 | triplet_acc=0.955 | lr=0.00400
Epoch  200 | loss=1.0979 | triplet_acc=0.990 | lr=0.00399
Epoch  300 | loss=0.2049 | triplet_acc=0.998 | lr=0.00399
Epoch  400 | loss=0.3134 | triplet_acc=0.998 | lr=0.00398
Epoch  500 | loss=0.0000 | triplet_acc=0.998 | lr=0.00398
Epoch  600 | loss=0.0000 | triplet_acc=1.000 | lr=0.00398
Epoch  700 | loss=0.0000 | triplet_acc=1.000 | lr=0.00397
Epoch  800 | loss=0.0000 | triplet_acc=0.985 | lr=0.00397
Epoch  900 | loss=1.7083 | triplet_acc=0.995 | lr=0.00396
Epoch 1000 | loss=0.6158 | triplet_acc=1.000 | lr=0.00396
Epoch 1100 | loss=0.0000 | triplet_acc=1.000 | lr=0.00396
Epoch 1200 | loss=0.0000 | triplet_acc=1.000 | lr=0.00395
Epoch 1300 | loss=0.0000 | triplet_acc=1.000 | lr=0.00395
Epoch 1400 | loss=0.0000 | triplet_acc=1.000 | lr=0.00394
Epoch 1500 | loss=0.0000 | triplet_acc=1.000 | lr=0.00394
Epoch 1600 | loss=0.0000 | triplet_acc=1.000 | lr=0.00394
Epoch 1700 | l

In [3]:
train_trajs, train_features, test_trajs, test_features = get_train_test_trajs_feats()
eval_fpe(model, train_trajs, train_features, test_trajs, test_features)

all_trajs shape: (900, 21, 97)
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis elemen

np.float64(0.06377325789405089)

# sirl expts

In [4]:
from utils import *
from sirl import *
from fpe import *
import numpy as np
import torch
import random
import json

OUTFILE = "061426_sirl_results.json"

# --- Seeds ---
# Fixed across the whole experiment: ensures the FPE eval set is identical
# for every SIRL model we evaluate. NEVER change this between conditions.
FPE_SPLIT_SEED = 42

# SIRL training seeds: vary across these to get error bars on FPE.
SIRL_SEEDS = [0, 1, 2]


def set_all_seeds(seed):
    """Set seeds for every source of randomness in SIRL training."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# --- Build FPE eval data ONCE, outside the seed loop ---
# Uses FPE_SPLIT_SEED so the split is deterministic and shared.
train_trajs, train_features, test_trajs, test_features = \
    get_train_test_trajs_feats(seed=FPE_SPLIT_SEED)


results = {}
for n_queries in range(100, 1100, 100):
    print(f"\n=== TRAINING SIRL FOR {n_queries} QUERIES ===")
    anchors, positives, negatives = load_data(n_queries)

    fpes_for_n = []
    for seed in SIRL_SEEDS:
        print(f"  -- seed {seed} --")
        set_all_seeds(seed)

        model, _ = train_sirl(
            anchors, positives, negatives,
            use_symmetric_loss=True,
        )

        fpe = eval_fpe(model, train_trajs, train_features, test_trajs, test_features)

        print(f"     FPE = {fpe:.4f}")
        fpes_for_n.append(fpe)

    fpes = np.array(fpes_for_n)
    results[n_queries] = {
        "mean": float(fpes.mean()),
        "std":  float(fpes.std(ddof=1)),
        "all":  fpes.tolist(),
        "seeds": SIRL_SEEDS,
    }
    print(f"  N={n_queries}: FPE = {fpes.mean():.4f} ± {fpes.std(ddof=1):.4f}")

    with open(OUTFILE, "w") as f:
        json.dump(results, f, indent=4)

print("\nDone.")
print(json.dumps(results, indent=2))

all_trajs shape: (10000, 21, 97)
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis elem

KeyboardInterrupt: 

# pca baseline

In [ ]:
from utils import *
from sirl import *
from fpe import *
import numpy as np
import torch
import random
import json

OUTFILE = "061426_pca_results.json"

# --- Seeds ---
# Fixed across the whole experiment: ensures the FPE eval set is identical
# for every SIRL model we evaluate. NEVER change this between conditions.
FPE_SPLIT_SEED = 42

# SIRL training seeds: vary across these to get error bars on FPE.
SIRL_SEEDS = [0, 1, 2]


def set_all_seeds(seed):
    """Set seeds for every source of randomness in SIRL training."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# --- Build FPE eval data ONCE, outside the seed loop ---
# Uses FPE_SPLIT_SEED so the split is deterministic and shared.
train_trajs, train_features, test_trajs, test_features = \
    get_train_test_trajs_feats(seed=FPE_SPLIT_SEED)


results = {}
for n_queries in range(100, 1100, 100):
    print(f"\n=== TRAINING SIRL FOR {n_queries} QUERIES ===")
    anchors, positives, negatives = load_data(n_queries)

    fpes_for_n = []
    for seed in SIRL_SEEDS:
        print(f"  -- seed {seed} --")
        set_all_seeds(seed)

        query_trajs = np.concatenate([anchors, positives, negatives], axis=0)
        pca = fit_pca(query_trajs, n_components=6, random_state=42)

        fpe = eval_pca_fpe(pca, train_trajs, train_features, test_trajs, test_features)
        
        print(f"     FPE = {fpe:.4f}")
        fpes_for_n.append(fpe)

    fpes = np.array(fpes_for_n)
    results[n_queries] = {
        "mean": float(fpes.mean()),
        "std":  float(fpes.std(ddof=1)),
        "all":  fpes.tolist(),
        "seeds": SIRL_SEEDS,
    }
    print(f"  N={n_queries}: FPE = {fpes.mean():.4f} ± {fpes.std(ddof=1):.4f}")

    with open(OUTFILE, "w") as f:
        json.dump(results, f, indent=4)

print("\nDone.")
print(json.dumps(results, indent=2))

all_trajs shape: (10000, 21, 97)
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis elem

# tversky

`tversky_tie_fbank_and_prototypes` [here](https://github.com/mdoumbouya/tversky-networks-iclr2026/blob/main/experiments/006-language-modeling/src/tversky_gpt2/modeling_tversky_gpt2.py) does feature bank Omega tying

In [ ]:
from tversky import nn as tnn

sim_layer = tnn.TverskySimilarity(
    embedding_dim=64,
    fbank_size=128,
    similarity_model='contrast',
    normalize=False
)

In [3]:
sim_layer

TverskySimilarity(
  (feature_bank): Embedding(128, 64)
)

In [ ]:
proj_layer = tnn.TverskyProjection(
    embedding_dim=64,
    class_count=10,
    fbank_size=128,
    similarity_model='contrast',
    normalize=False
)

In [ ]:
from utils import *
from tversky_sirl import *
from fpe import *
import numpy as np
import torch
import random
import json

OUTFILE = "061426_tversky_results.json"

# --- Seeds ---
# Fixed across the whole experiment: ensures the FPE eval set is identical
# for every SIRL model we evaluate. NEVER change this between conditions.
FPE_SPLIT_SEED = 42

# SIRL training seeds: vary across these to get error bars on FPE.
SIRL_SEEDS = [0, 1, 2, 3, 4]


def set_all_seeds(seed):
    """Set seeds for every source of randomness in SIRL training."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# --- Build FPE eval data ONCE, outside the seed loop ---
# Uses FPE_SPLIT_SEED so the split is deterministic and shared.
train_trajs, train_features, test_trajs, test_features = \
    get_train_test_trajs_feats(seed=FPE_SPLIT_SEED)

results = {}
for n_queries in range(100, 1100, 100):
    print(f"\n=== TRAINING TVERSKY SIRL FOR {n_queries} QUERIES ===")
    anchors, positives, negatives = load_data(n_queries)

    fpes_for_n = []
    for seed in SIRL_SEEDS:
        print(f"  -- seed {seed} --")
        set_all_seeds(seed)

        model, _ = train_tversky_sirl(
            anchors, positives, negatives,
            use_symmetric_loss=True,
        )

        fpe = eval_fpe(model, train_trajs, train_features, test_trajs, test_features)
        
        print(f"     FPE = {fpe:.4f}")
        fpes_for_n.append(fpe)

    fpes = np.array(fpes_for_n)
    results[n_queries] = {
        "mean": float(fpes.mean()),
        "std":  float(fpes.std(ddof=1)),
        "all":  fpes.tolist(),
        "seeds": SIRL_SEEDS,
    }
    print(f"  N={n_queries}: FPE = {fpes.mean():.4f} ± {fpes.std(ddof=1):.4f}")

    with open(OUTFILE, "w") as f:
        json.dump(results, f, indent=4)

print("\nDone.")
print(json.dumps(results, indent=2))

pybullet build time: Oct 21 2025 16:30:45


all_trajs shape: (900, 21, 97)
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis elemen

## sweep feature bank size

In [3]:
from utils import *
from tversky_sirl import *
from fpe import *
import numpy as np
import torch
import random
import json

# --- Sweep config ---
N_QUERIES = 1000
FBANK_SIZES = [4, 16, 64, 256]
OUTFILE_TEMPLATE = "061426_tversky_{fbank}_results.json"

# --- Seeds ---
# Fixed across the whole experiment: ensures the FPE eval set is identical
# for every SIRL model we evaluate. NEVER change this between conditions.
FPE_SPLIT_SEED = 42

# SIRL training seeds: vary across these to get error bars on FPE.
SIRL_SEEDS = [0, 1, 2, 3, 4]


def set_all_seeds(seed):
    """Set seeds for every source of randomness in SIRL training."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# --- Build FPE eval data ONCE, outside all loops ---
# Uses FPE_SPLIT_SEED so the split is deterministic and shared.
train_trajs, train_features, test_trajs, test_features = \
    get_train_test_trajs_feats(seed=FPE_SPLIT_SEED)


# --- Load triplet data ONCE (same for all fbank sizes) ---
print(f"Loading {N_QUERIES} queries...")
anchors, positives, negatives = load_data(N_QUERIES)


# --- Sweep over feature bank sizes ---
all_results = {}
for fbank_size in FBANK_SIZES:
    outfile = OUTFILE_TEMPLATE.format(fbank=fbank_size)
    print(f"\n=== TRAINING TVERSKY SIRL | n_queries={N_QUERIES} | fbank_size={fbank_size} ===")

    fpes_for_fbank = []
    for seed in SIRL_SEEDS:
        print(f"  -- seed {seed} --")
        set_all_seeds(seed)

        model, _ = train_tversky_sirl(
            anchors, positives, negatives,
            use_symmetric_loss=True,
            fbank_size=fbank_size,
        )

        fpe = eval_fpe(model, train_trajs, train_features, test_trajs, test_features)

        print(f"     FPE = {fpe:.4f}")
        fpes_for_fbank.append(fpe)

    fpes = np.array(fpes_for_fbank)
    results = {
        "n_queries": N_QUERIES,
        "fbank_size": fbank_size,
        "mean": float(fpes.mean()),
        "std":  float(fpes.std(ddof=1)),
        "all":  fpes.tolist(),
        "seeds": SIRL_SEEDS,
    }
    print(f"  fbank={fbank_size}: FPE = {fpes.mean():.4f} ± {fpes.std(ddof=1):.4f}")

    with open(outfile, "w") as f:
        json.dump(results, f, indent=4)
    print(f"  wrote: {outfile}")

    all_results[fbank_size] = results

print("\nDone.")
print(json.dumps(all_results, indent=2))

all_trajs shape: (10000, 21, 97)
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis elem

# random baseline

In [2]:
from utils import *
from sirl import *
from fpe import *
import numpy as np
import torch
import random
import json

OUTFILE = "061426_random_results.json"

# --- Seeds ---
# Fixed across the whole experiment: ensures the FPE eval set is identical
# for every SIRL model we evaluate. NEVER change this between conditions.
FPE_SPLIT_SEED = 42

# SIRL training seeds: vary across these to get error bars on FPE.
SIRL_SEEDS = [0, 1, 2, 3, 4]


def set_all_seeds(seed):
    """Set seeds for every source of randomness in SIRL training."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# --- Build FPE eval data ONCE, outside the seed loop ---
# Uses FPE_SPLIT_SEED so the split is deterministic and shared.
train_trajs, train_features, test_trajs, test_features = \
    get_train_test_trajs_feats(seed=FPE_SPLIT_SEED)

input_dim = 2037
latent_dim=6
hidden_dim=1024

results = {}
for n_queries in range(100, 1100, 100):
    

    print(f"\n=== RANDOM SIRL ignoring {n_queries} QUERIES ===")
    # anchors, positives, negatives = load_data(n_queries)

    fpes_for_n = []
    for seed in SIRL_SEEDS:
        print(f"  -- seed {seed} --")
        set_all_seeds(seed)
        model = SIRL(input_dim=input_dim, hidden_dim=hidden_dim, latent_dim=latent_dim)

        fpe = eval_fpe(model, train_trajs, train_features, test_trajs, test_features)

        print(f"     FPE = {fpe:.4f}")
        fpes_for_n.append(fpe)

    fpes = np.array(fpes_for_n)
    results[n_queries] = {
        "mean": float(fpes.mean()),
        "std":  float(fpes.std(ddof=1)),
        "all":  fpes.tolist(),
        "seeds": SIRL_SEEDS,
    }
    print(f"  N={n_queries}: FPE = {fpes.mean():.4f} ± {fpes.std(ddof=1):.4f}")

    with open(OUTFILE, "w") as f:
        json.dump(results, f, indent=4)

print("\nDone.")
print(json.dumps(results, indent=2))

all_trajs shape: (10000, 21, 97)
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis elem